# Pytorch의 nn.Embedding
- Pytorch의 Embedding Layer는 word2vec과 마찬가지로 word embedding vector를 찾는 **Lookup Table**이다.
    - 단어의 **정수의 고유 index**가 입력으로 들어오면 Embedding Layer의 **그 index의 Vector**를 출력한다.
    - 모델이 학습되는 동안 모델이 풀려는 문제에 맞는 값으로 Embedding Layer의 vector들이 업데이트 된다.
    - Word2Vec의 embedding vector 학습을 nn.Embedding은 자신이 포함된 모델을 학습 하는 과정에서 한다고 생각하면 된다.

In [1]:
import torch
import torch.nn as nn

embedding_model = nn.Embedding(
    num_embeddings=20000, # vocab_size(어휘사전의 어휘개수): 몇개 단어(토큰)에 대한 Embedding vector를 만들지 생각
    embedding_dim=200,   # Embedding vector의 차원수
    padding_idx=0,        # padding 토큰의 index. 실제 padding 값에 맞춰라. padding 토큰은 글자수를 맞추기 위해 채우는 값(0) 그래서 embedding값을 학습할 필요가 없다.
)

# 20000 x 200
# nn.GRU(input_szie=200) # Embedding vector의 차원수가 input사이즈

In [2]:
# 파라미터 확인
weight = embedding_model.weight
weight.shape
weight[0] # 0번 토큰(단어)의 embedding vector값을 조회(0번은 padding_idx=0으로 인해 전부 0으로 나온다)
weight[1]

tensor([-0.0305, -1.0569,  0.4750,  0.8720, -0.7834, -0.4093,  1.7651, -1.5706,
         0.7346,  1.1622,  0.5462,  1.3934, -0.6629, -0.1296, -0.5690,  1.3298,
        -0.3780, -0.7386,  1.8923, -0.9735,  0.0265, -0.0069,  0.4603,  0.0355,
        -0.5501, -1.0400,  0.2889,  0.6712, -0.8524,  0.5652,  0.7299, -1.0786,
        -0.9222,  0.3107,  1.9614,  1.2748,  1.2128,  0.0161, -0.9846,  1.3671,
         0.0746,  1.0957,  1.6068,  0.2972,  0.6306, -0.0187,  0.9121, -1.0433,
        -0.5321,  1.9018,  1.0236,  0.0276, -0.4949,  1.1029, -1.9268,  1.3037,
         0.3818, -0.2324,  1.0512,  0.2722,  1.3105,  0.0288,  0.3243, -0.7432,
        -0.6586, -1.1678,  1.3601, -1.9527,  0.7260, -1.6551, -0.2367,  0.0057,
         0.6759, -0.7661, -0.2553,  0.7205,  1.6613, -0.9989, -0.4623,  0.2644,
        -0.0285, -1.1461,  0.3008,  0.8683,  0.2877,  0.3269,  0.1824,  0.0061,
        -1.4828,  1.0287, -2.0107, -1.9464, -1.2243, -0.8344, -0.7421,  0.5514,
        -0.6521,  0.9645,  0.1793, -1.15

In [3]:
# 예시: "나는 -30 어제 -100 밥을 -600, 먹었다 -7200 . -5"
# # 문장을 tokenizer를 통해 토큰화한 결과
# 한문장인 경우, batch_size가 1
doc_token_ids=torch.tensor([[30, 100, 600, 7200, 5]], dtype=torch.int64)
# 여러문장인 경우, batch_size가 3
# doc_token_ids=torch.tensor([[30, 100, 600, 7200, 5],[30, 100, 600, 7200, 5],[30, 100, 600, 7200, 5]], dtype=torch.int64)

doc_embedding_vector = embedding_model(doc_token_ids)

doc_embedding_vector.shape # [1: batch_size-1문장, 5: seq_len-토큰개수, 200: embedding vector]
# [1, 5, 200], -> [batch_size, seq_len, embedding vector]

torch.Size([1, 5, 200])

# 네이버 영화 댓글 감성분석(Sentiment Analysis)

## 감성분석(Sentiment Analysis) 이란
입력된 텍스트가 **긍적적인 글**인지 **부정적인**인지 또는 **중립적인** 글인지 분석하는 것을 감성(감정) 분석이라고 한다.   
이를 통해 기업이 고객이 자신들의 기업 또는 제품에 대해 어떤 의견을 가지고 있는지 분석한다.

# Dataset, DataLoader 생성

## Korpora에서 Naver 영화 댓글 dataset 가져오기
- https://ko-nlp.github.io/Korpora/ko-docs/corpuslist/nsmc.html
- http://github.com/e9t/nsmc/
    - input: 영화댓글
    - output: 0(부정적댓글), 1(긍정적댓글)
### API
- **corpus 가져오기**
    - `Korpora.load('nsmc')`
- **text/label 조회**
    - `corpus.get_all_texts()` : 전체 corpus의 text들을 tuple로 반환
    - `corpus.get_all_labels()`: 전체 corpus의 label들을 list로 반환
- **train/test set 나눠서 조회**
    - `corpus.train`
    - `corpus.test`
    - `LabeledSentenceKorpusData` 객체에 text와 label들을 담아서 제공.
        - `LabeledSentenceKorpusData.texts`: text들 tuple로 반환.
        - `LabeledSentenceKorpusData.labels`: label들 list로 반환.

## 데이터 로딩

In [4]:
from Korpora import Korpora
corpus = Korpora.load('nsmc')


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:01, 11.5MB/s]                            
[nsmc] download ratings_test.txt: 4.90MB [00:00, 11.2MB/s]                            


In [5]:
all_inputs = corpus.get_all_texts()  # X: 댓글
all_labels = corpus.get_all_labels() # y: label (0: 부정적, 1: 긍정적)

In [6]:
len(all_inputs)

200000

In [7]:
print(type(corpus.train))
corpus.train

<class 'Korpora.korpora.LabeledSentenceKorpusData'>


NSMC.train: size=150000
  - NSMC.train.texts : list[str]
  - NSMC.train.labels : list[int]

In [8]:
corpus.train.texts[:5]
corpus.train.labels[:5]

[0, 1, 0, 0, 1]

In [9]:
corpus.test

NSMC.test: size=50000
  - NSMC.test.texts : list[str]
  - NSMC.test.labels : list[int]

## 토큰화
1. 형태소 단위 token화
    - konlpy로 token화 한 뒤 다시 한 문장으로 만든다.
2. 1에서 처리한 corpus를 BPE 로 token화
   
### 전처리 함수

#### 형태소 단위 분절

In [10]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [11]:
# from konlpy.tag import Okt
from kiwipiepy import Kiwi
import string
import re

kiwi = Kiwi()
def text_preprocessing(text):
    """
    1. 영문 -> 소문자로 변환
    2. 구두점 제거
    3. 형태소 기반 토큰화
    4. 형태소로 토큰화 한 뒤 다시 하나의 문자열로 묶어서 반환.
    """
    text = text.lower()
    text = re.sub(rf"[{string.punctuation}]", ' ', text) # 구두점(특수문자)들을 공백으로 바꾼다
    text = [token.lemma for token in kiwi.tokenize(text)]
    return ' '.join(text)

In [12]:
text_preprocessing(all_inputs[100])

'신카이 마코토 의 작화 와 미유 와 하나카나 가 연기 를 잘 하다 어 주다 어서 더 대박 이다 였 다'

In [14]:
train_texts = corpus.train.texts
train_inputs = [text_preprocessing(txt) for txt in train_texts]

test_texts = corpus.test.texts
test_inputs = [text_preprocessing(txt) for txt in test_texts]

train_labels = corpus.train.labels
test_labels = corpus.test.labels

In [17]:
# 데이터셋을 피클로 저장
import os
# os.makedirs('data/nsmc')

train_data = {"text": train_inputs, "label":train_labels}
test_data = {"text": test_inputs, "label":test_labels}

import pickle
with open("data/nsmc/preprocessing_train.pkl", "wb") as fo:
    pickle.dump(train_data, fo)

with open("data/nsmc/preprocessing_test.pkl", "wb") as fo:
    pickle.dump(test_data, fo)
    

In [ ]:
all_inputs = train_inputs + test_inputs # list + list
# train/test set의 댓글들을 합치기 -> 토크나이저(어휘사전) 생성을 위해

### 토큰화
- Subword 방식 토큰화 적용
- Byte Pair Encoding 방식으로 huggingface tokenizer 사용
    - BPE: 토큰을 글자 단위로 나눈뒤 가장 자주 등장하는 글자 쌍(byte paire)를 찾아 합친뒤 어휘사전에 추가한다.
    - https://huggingface.co/docs/tokenizers/quicktour
    - `pip install tokenizers`

In [27]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

vocab_size= 30_000

tokenizer = Tokenizer(
    BPE(unk_token="<unk>")
)

tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size = vocab_size,
    min_frequency=5,
    special_tokens=["<pad>", "<unk>"],
    continuing_subword_prefix="##",
    # 시작 subword는 그대로. 연결 subword 앞에는 ##을 붙인다.
    # cowork : co, ##work
)

tokenizer.train_from_iterator(all_inputs, trainer = trainer)

# 학습데이터가 파일: tokenizer.train(["파일 경로"])
# 학습 데이터가 메모리에 iterable타입으로 있는 경우: tokenizer.train_from_iterator()

In [28]:
# 어휘사전 크기
tokenizer.get_vocab_size()

30000

In [29]:
# 저장
tokenizer.save("saved_models/nsmc_bpe_tokenizer.json")
# 불러오기: load_tokenizer = Tokenizer.from_frile("경로")

In [31]:
# 인코딩 테스트
idx = 0
print(all_inputs[idx])

encode = tokenizer.encode(all_inputs[idx])

print(encode.tokens)
print(encode.ids)

아 더빙.. 진짜 짜증나네요 목소리
['아', '더빙', '..', '진짜', '짜증나네요', '목소리']
[2109, 6728, 6050, 6068, 19545, 7434]


In [32]:
tokenizer.decode(encode.ids)

'아 더빙 .. 진짜 짜증나네요 목소리'

## Dataset, DataLoader 생성

In [33]:
# Dataset - Raw 데이터 셋에서 학습할 때 필요한 데이터를 하나씩 제공 역할
#           subscriptable 타입의 클래스로 구현 (__len__(), __getitem__(index) 둘을 구현)
#           Dataset객체[0]  0번 학습데이터를 제공. (x[0],y[0])
# DataLoader - Batch 단위로 묶어서 데이터를 제공

import torch
from torch.utils.data import Dataset, DataLoader

class NSMCDataset(Dataset):
    def __init__(self, texts, labels, max_length, tokenizer):
        """
        texts: list - 댓글 리스트. 리스트에 댓글들을 담아서 받는다. ["댓글", "댓글", ...]
        labels: list - Label 리스트. (댓글의 긍부정 여부 - 긍정: 1, 부정: 0)
        max_length: 개별 댓글의 token 개수. 모든 댓글의 토큰수를 max_length에 맞춘다.
        tokenizer: Tokenizer
        """
        self.max_length = max_length
        self.tokenizer = tokenizer
        self.labels= labels
        self.texts = [ self.__pad_token_sequences(tokenizer.encode(txt).ids) for txt in texts]
        # 댓글 -> 토큰 ID, max_length 크기에 맞춤(토큰 수가 적으면 <pad>추가, 많으면 잘라내기)

    ###########################################################################################
    # id로 구성된 개별 문장 token list를 받아서 패딩 추가 [20, 2, 1] => [20, 2, 1, 0, 0, 0, ..]
    ############################################################################################
    def __pad_token_sequences(self, token_sequences):
        """
        token id로 구성된 개별 문서(댓글)의 token_id list를 받아서 max_length 길이에 맞추는 메소드
        max_length 보다 토큰수가 적으면 <pad> 토큰 추가, 많으면 max_length 크기로 줄인다.
            ex) max_length=5 이고 pad토큰 id가 0이라면
                [20, 2, 1] => [20, 2, 1, 0, 0, 0]
                [20, 21, 30, 34, 60, 17, 21, 33] -> [20, 21, 30, 34, 60]
        """
        pass
        
    def __len__(self):
        # 총 Dataset의 개수를 반환
        return len(self.texts)

    def __getitem__(self, idx):
        """
        idx 번째 text와 label을 학습 가능한 type으로 변환해서 반환
        Parameter
            idx: int 조회할 index
        Return
            tuple: (torch.LongTensor, torch.FloatTensor) - 댓글 토큰_id 리스트, 정답 Label
        """
        pass
    

# 모델링
- Embedding Layer를 이용해 Word Embedding Vector를 추출한다.
- LSTM을 이용해 Feature 추출
- Linear + Sigmoid로 댓글 긍정일 확률 출력
  
![outline](figures/rnn/RNN_outline.png)

## 모델 정의

## 모델 생성

## 학습

### Train/Test 함수 정의

### Train

## 모델저장

# 서비스

## 전처리 함수들

In [ ]:
from konlpy.tag import Okt

okt = Okt()
def text_preprocessing(text):
    
    text = text.lower()
    text = re.sub(f"[{string.punctuation}]+", ' ', text)
    return ' '.join(okt.morphs(text, stem=True))

In [ ]:
def pad_token_sequences(token_sequences, max_length):
    """padding 처리 메소드."""
    pad_token = tokenizer.token_to_id('[PAD]')  
    seq_length = len(token_sequences)           
    result = None
    if seq_length > max_length:                 
        result = token_sequences[:max_length]
    else:                                            
        result = token_sequences + ([pad_token] * (max_length - seq_length))
    return result

In [ ]:
def predict_data_preprocessing(text_list):
    """
    모델에 입력할 수있는 input data를 생성
    Parameter:
        text_list: list - 추론할 댓글리스트
    Return
        torch.LongTensor - 댓글 token_id tensor
    """
   
    pass

## 추론

In [ ]:
comment_list = ["아 진짜 재미없다.", "여기 식당 먹을만 해요", "이걸 영화라고 만들었냐?", "기대 안하고 봐서 그런지 괜찮은데.", "이걸 영화라고 만들었나?", "아! 뭐야 진짜.", "재미있는데.", "연기 짱 좋아. 한번 더 볼 의향도 있다.", "뭐 그럭저럭"]